# Certamen de laboratorio · Minería de Datos · Unidad II

**martes 27 de octubre de 2026** · **individual** · 120 minutos · 100 puntos · 40 % de la nota de práctica

**Nombre:** _escriba aquí su nombre_ · **RUT:** _escriba aquí su RUT_

## Antes de empezar (lea esto primero)

1. **Archivo → Guardar una copia en Drive.** Trabaje siempre en su copia.
2. Ejecute la **celda ▶ entregada** (la primera de código). Debe ver: *Datos cargados: 200 fichas (160 de entrenamiento y 40 de prueba), 300 pedidos, 40 semanas y 60 estudiantes.*
3. Cada tarea tiene siempre las mismas partes, en este orden:

| Ícono | Qué es | Qué hace usted |
|:-:|---|---|
| 📖 | **Contexto** | Entender el problema del negocio |
| 💡 | **Concepto clave** | Repasar la idea que hay detrás |
| 🎯 | **Qué hacer** | Seguir los pasos |
| 📦 🔍 | **Qué va a obtener y qué representa** | Saber de antemano qué resultado buscar y qué significa para el caso |
| 🧩 | **Función con su firma** | Completar **solo el cuerpo**. No cambie el nombre ni los parámetros. El docstring dice qué recibe, qué devuelve y trae un ejemplo |
| 🧪 | **Prueba** con un ejemplo mínimo | Ejecutarla. Debe dar **exactamente** el resultado indicado, que se explica al lado. Si no da, corrija la función antes de seguir |
| 📊 | **Análisis** con los datos del laboratorio | Escribir el código que pide la tarea |
| ✅ | **Resultado esperado** | Comparar: dice **qué debe mostrar** su celda 📊 y en qué formato |
| ✍️ | **Respuesta** | Responder con **sus propios números** |

4. **De los datos al conocimiento.** Cada tarea termina con una pregunta 🧠. El curso trabaja con una escalera de tres peldaños:

| Peldaño | Qué es | Ejemplo en la clínica |
|---|---|---|
| **Dato** | Un hecho suelto | «La ficha 1023 recibió correo, vive lejos y no asistió» |
| **Información** | Datos procesados que muestran un patrón | «El árbol acierta 9 de cada 10 fichas de prueba» |
| **Conocimiento** | Información convertida en una **decisión** | «Hay que llamar a quienes reciben correo y viven lejos» |

   Su código produce **información**; sus respuestas 🧠 deben llegar al **conocimiento**.

5. Puede usar **todos los recursos** (apuntes, guías, internet, IA). Pero se evalúa que **interprete sus resultados**: una respuesta genérica, sin sus números, no tiene puntaje.
6. Tiempo sugerido: **Parte I** hasta el minuto 40, **Parte II** hasta el 80 y **Parte III** hasta el 120.
7. Al terminar: **Archivo → Descargar → Descargar .ipynb**, y súbalo al **Aula Virtual** como `Apellido_Nombre.ipynb`.

| Parte | Caso | Tareas | Puntos |
|---|---|:-:|:-:|
| I | Clínica veterinaria: ZeroR, OneR y árbol | 1 a 3 | 35 |
| II | Ferretería online: soporte, confianza, lift y regresión | 4 a 6 | 35 |
| III | Liceo: k-means por rondas y sus gráficas | 7 y 8 | 30 |

In [ ]:
# ▶ CELDA ENTREGADA — no la modifique. Importa las librerías y carga los datos.
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

RUTA = "https://raw.githubusercontent.com/de-mag-ia/diguillin-datos/main/certamen-lab/"
clinica = pd.read_csv(RUTA + "lab_clinica.csv")              # Parte I
pedidos = pd.read_csv(RUTA + "lab_pedidos.csv")              # Parte II (Apriori)
semanas = pd.read_csv(RUTA + "lab_semanas.csv")              # Parte II (regresión)
liceo = pd.read_csv(RUTA + "lab_liceo.csv")                  # Parte III

entrenamiento = clinica[clinica["Conjunto"] == "entrenamiento"]
prueba = clinica[clinica["Conjunto"] == "prueba"]
canastas = pedidos.groupby("Pedido")["Producto"].apply(set).tolist()          # un conjunto por pedido
puntos = list(zip(liceo["Estudio"], liceo["Actividad"]))                     # pares (x, y)

ATRIBUTOS = ["Especie", "Recordatorio", "Distancia"]
PRODUCTOS = ['Pintura', 'Brocha', 'Rodillo', 'Lija', 'Guantes', 'Huincha']


def acierto(predicciones):
    """Proporción de fichas de PRUEBA bien predichas (entre 0 y 1). Recibe una predicción por ficha de prueba."""
    return (prueba["Asistio"].values == pd.Series(predicciones).values).mean()

ESTRELLAS_INICIO = [(1, 9), (2, 1), (3, 3)]

print(f"Datos cargados: {len(clinica)} fichas ({len(entrenamiento)} de entrenamiento y {len(prueba)} de prueba), "
      f"{len(canastas)} pedidos, {len(semanas)} semanas y {len(puntos)} estudiantes.")

---
# Parte I · Clínica veterinaria (35 puntos)

La clínica quiere predecir **qué dueños no traerán a su mascota al control de vacunas**. La tabla `clinica` tiene **200 fichas** con:

| Columna | Valores |
|---|---|
| `Ficha` | número de ficha (1001 a 1200) |
| `Especie` | Perro · Gato |
| `Recordatorio` | WhatsApp · Correo · Ninguno |
| `Distancia` | Cerca · Lejos |
| `Asistio` | **Sí · No** ← la clase que se quiere predecir |
| `Conjunto` | `entrenamiento` (160 fichas, para **construir** los modelos) · `prueba` (40 fichas, para **evaluarlos**) |

Ya están separadas en las variables **`entrenamiento`** y **`prueba`**. Los modelos se construyen **solo** con `entrenamiento`.

## Tarea 1 · ZeroR: la línea base (8 puntos)

**📖 Contexto.** Antes de usar un modelo, la clínica quiere saber cuánto acierta **sin analizar nada**.

> **💡 Concepto clave.** **ZeroR** predice siempre la clase que más se repite. Su acierto es la **línea base**: el mínimo que cualquier modelo debe superar.

### 🎯 Qué hacer

1. Complete `zeror()` siguiendo las pistas que trae adentro y ejecute la prueba 🧪.
2. Complete los `___` de la celda 📊.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| Lo que predice ZeroR, sus **errores en entrenamiento** y su **acierto en prueba** (un número entre 0 y 1). | Ese acierto es **la vara para medir** los demás modelos: lo que se logra sin analizar ningún dato. |

In [ ]:
# 🧩 Tarea 1 · complete el cuerpo de la(s) función(es)
def zeror(df, clase):
    """ZeroR: predice SIEMPRE la clase que más se repite.

    Recibe:   df    → la tabla (DataFrame)
              clase → el nombre de la columna que se quiere predecir, aquí "Asistio"
    Devuelve: (clase_mayoritaria, errores)
              errores = cuántas filas NO son de la clase mayoritaria (ahí ZeroR se equivoca)
    Ejemplo:  si hay 5 filas «Sí» y 3 «No»  →  ("Sí", 3)
    """
    # Pista 1: cuente cuántas filas hay de cada clase   →  conteo = df[clase].value_counts()
    #          (queda ordenado de mayor a menor)
    # Pista 2: la clase mayoritaria es la primera          →  conteo.index[0]
    # Pista 3: los errores son todas las demás filas       →  len(df) - conteo.iloc[0]
    # Pista 4: devuelva las dos cosas                      →  return mayoritaria, errores
    pass

In [ ]:
# 🧪 Prueba de la tarea 1 con un ejemplo mínimo: ejecútela sin cambiarla
mini = pd.DataFrame({
    "Recordatorio": ["WhatsApp", "WhatsApp", "WhatsApp", "Correo", "Correo", "Correo", "Ninguno", "Ninguno"],
    "Distancia":    ["Cerca",    "Lejos",    "Cerca",    "Cerca",  "Cerca",  "Lejos",  "Cerca",   "Lejos"],
    "Asistio":      ["Sí",       "Sí",       "Sí",       "Sí",     "Sí",     "No",     "No",      "No"]})
c, e = zeror(mini, "Asistio")
print(c, e)
# ✅ Debe mostrar: Sí 3

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`Sí 3`**: en las 8 fichas del ejemplo hay 5 «Sí» y 3 «No»: ZeroR predice «Sí» y falla en las 3 «No». Si le da 5, está contando aciertos.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 1 · complete los ___ y ejecute
clase_zr, errores_zr = zeror(___, "Asistio")          # use las fichas de entrenamiento
print("ZeroR predice:", clase_zr, "| errores:", errores_zr)
acierto_zr = acierto([clase_zr] * len(prueba))          # ZeroR predice lo mismo para las 40 fichas de prueba
print("Acierto en prueba:", round(acierto_zr, 3))

**✅ Compare:** su celda 📊 debe mostrar lo que predice ZeroR, sus **errores en entrenamiento** y su **acierto en prueba** (un número entre 0 y 1).

### ✍️ Responda

**1. Sus resultados:** ¿Qué predice ZeroR y qué acierto obtiene en prueba?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Si la clínica supusiera que todos vienen, ¿en cuántas de cada 10 mascotas se equivocaría?

_Su respuesta:_

## Tarea 2 · OneR: ¿qué dato explica más la asistencia? (12 puntos)

**📖 Contexto.** La clínica quiere saber cuál de tres datos (especie, recordatorio o distancia) **explica mejor** si el dueño viene.

> **💡 Concepto clave.** **OneR** prueba cada dato por separado: para cada valor predice la clase más frecuente, y cuenta los errores. Gana el dato con **menos errores**.

### 🎯 Qué hacer

1. Complete `oner()` siguiendo las pistas (puede usar su `zeror()`) y ejecute la prueba 🧪.
2. Complete los `___` de la celda 📊: errores de cada atributo, el ganador y su acierto en prueba.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| Los **errores de los 3 atributos**, el **ganador** con **su regla** (un valor → una clase) y su **acierto en prueba**. | El ganador es **el dato que más ayuda a predecir** la asistencia; su regla ya es una política simple para la clínica. |

In [ ]:
# 🧩 Tarea 2 · complete el cuerpo de la(s) función(es)
def oner(df, atributo, clase):
    """OneR con UN atributo: una regla por cada valor del atributo.

    Para cada valor del atributo: la regla es la clase que más se repite en ese valor, y los
    errores son las filas de ese valor que tienen la otra clase. Se suman los errores.
    Recibe:   df, atributo (p. ej. "Recordatorio"), clase ("Asistio")
    Devuelve: (reglas, errores)
              reglas = diccionario {valor: clase predicha}
    Ejemplo:  si con "Correo" hay 2 «Sí» y 1 «No», su regla es "Correo" → "Sí" y suma 1 error.
              Devuelve algo como ({"Correo": "Sí", "Ninguno": "No", "WhatsApp": "Sí"}, 1)
    """
    reglas = {}
    errores = 0
    for valor, grupo in df.groupby(atributo):     # un grupo de filas por cada valor del atributo
        # Pista: zeror(grupo, clase) le da la clase mayoritaria del grupo y sus errores
        # 1. guarde la regla de ese valor:   reglas[valor] = ...
        # 2. sume los errores de ese valor:  errores = errores + ...
        pass
    return reglas, errores

In [ ]:
# 🧪 Prueba de la tarea 2 con un ejemplo mínimo: ejecútela sin cambiarla
mini = pd.DataFrame({
    "Recordatorio": ["WhatsApp", "WhatsApp", "WhatsApp", "Correo", "Correo", "Correo", "Ninguno", "Ninguno"],
    "Distancia":    ["Cerca",    "Lejos",    "Cerca",    "Cerca",  "Cerca",  "Lejos",  "Cerca",   "Lejos"],
    "Asistio":      ["Sí",       "Sí",       "Sí",       "Sí",     "Sí",     "No",     "No",      "No"]})
reglas, e = oner(mini, "Recordatorio", "Asistio")
print(dict(sorted(reglas.items())), e)
# ✅ Debe mostrar: {'Correo': 'Sí', 'Ninguno': 'No', 'WhatsApp': 'Sí'} 1

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`{'Correo': 'Sí', 'Ninguno': 'No', 'WhatsApp': 'Sí'} 1`**: Correo: 2 Sí y 1 No (regla Sí, 1 error); Ninguno: 2 No (regla No, 0 errores); WhatsApp: 3 Sí (regla Sí, 0 errores). Total: 1 error.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 2 · complete los ___ y ejecute
for atributo in ATRIBUTOS:
    reglas, errores = oner(entrenamiento, ___, "Asistio")
    print(atributo, "→", errores, "errores")
ganador = ___                                            # escriba entre comillas el atributo con MENOS errores
reglas_ganador, _ = oner(entrenamiento, ganador, "Asistio")
print("Regla del ganador:", reglas_ganador)
pred_oner = [reglas_ganador[v] for v in prueba[ganador]]  # la regla aplicada a cada ficha de prueba
print("Acierto en prueba:", round(acierto(pred_oner), 3))

**✅ Compare:** su celda 📊 debe mostrar los **errores de los 3 atributos**, el **ganador** con **su regla** (un valor → una clase) y su **acierto en prueba**.

### ✍️ Responda

**1. Sus resultados:** ¿Qué atributo gana, cuál es su regla y cuánto acierta en prueba comparado con ZeroR? ¿Qué atributo no le gana a ZeroR?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Según la regla ganadora, ¿qué le recomendaría a la clínica sobre **cómo enviar los recordatorios**?

_Su respuesta:_

## Tarea 3 · Árbol: una segunda pregunta donde hace falta (15 puntos)

**📖 Contexto.** La regla de OneR se equivoca casi siempre en **una misma rama**. El árbol agrega **otra pregunta solo en esa rama**.

> **💡 Concepto clave.** Una **rama impura** mezcla «Sí» y «No». En ella se busca la segunda pregunta aplicando OneR **solo a las fichas de esa rama**. Las demás ramas quedan igual.

### 🎯 Qué hacer

1. No hay función nueva: use `zeror()`, `oner()` y la función entregada `predecir_arbol()`.
2. Complete los `___` de la celda 📊 en tres pasos: (1) la rama con más errores, (2) la segunda pregunta y (3) el acierto del árbol.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| Los **errores de cada rama**, la **rama impura**, la **segunda pregunta** y el **acierto del árbol en prueba**. | El árbol es un **protocolo de dos preguntas**. Si acierta más que OneR, combinar dos datos predice mejor que uno solo. |

In [ ]:
# ▶ FUNCIÓN ENTREGADA — no la modifique. Clasifica una ficha con el árbol de dos preguntas.
def predecir_arbol(fila, raiz, reglas_raiz, rama, atributo2, reglas2):
    """Si la ficha cae en la rama impura, decide la 2.ª pregunta; si no, decide la regla de la raíz.

    Uso: predecir_arbol(fila, ganador, reglas_ganador, rama, atributo2, reglas2)
         con ganador y reglas_ganador de la tarea 2, y rama, atributo2 y reglas2 de esta tarea.
    """
    if fila[raiz] == rama:
        return reglas2.get(fila[atributo2], reglas_raiz[rama])
    return reglas_raiz[fila[raiz]]

In [ ]:
# 🧪 Prueba de la tarea 3 con un ejemplo mínimo: ejecútela sin cambiarla
mini = pd.DataFrame({
    "Recordatorio": ["WhatsApp", "WhatsApp", "WhatsApp", "Correo", "Correo", "Correo", "Ninguno", "Ninguno"],
    "Distancia":    ["Cerca",    "Lejos",    "Cerca",    "Cerca",  "Cerca",  "Lejos",  "Cerca",   "Lejos"],
    "Asistio":      ["Sí",       "Sí",       "Sí",       "Sí",     "Sí",     "No",     "No",      "No"]})
rama_mini = mini[mini["Recordatorio"] == "Correo"]
c, e = zeror(rama_mini, "Asistio")
r, e2 = oner(rama_mini, "Distancia", "Asistio")
print(c, e, dict(sorted(r.items())), e2)
# ✅ Debe mostrar: Sí 1 {'Cerca': 'Sí', 'Lejos': 'No'} 0

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`Sí 1 {'Cerca': 'Sí', 'Lejos': 'No'} 0`**: en el ejemplo, la rama Correo es la impura (2 Sí y 1 No: 1 error). Dentro de ella, Distancia separa sin errores: las 2 fichas Cerca asistieron y la Lejos no.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 3 · complete los ___ y ejecute
# (1) Errores de cada rama: la que tiene MÁS errores es la impura
for valor in sorted(entrenamiento[ganador].unique()):
    fichas_rama = entrenamiento[entrenamiento[ganador] == valor]
    print(valor, "→", zeror(fichas_rama, "Asistio")[1], "errores")
rama = ___                                               # escriba entre comillas la rama con más errores

# (2) Segunda pregunta: OneR solo con las fichas de esa rama
fichas_rama = entrenamiento[entrenamiento[ganador] == rama]
for atributo in ATRIBUTOS:
    if atributo != ganador:
        print(atributo, "→", oner(___, atributo, "Asistio")[1], "errores dentro de la rama")
atributo2 = ___                                          # el atributo con menos errores dentro de la rama
reglas2, _ = oner(fichas_rama, atributo2, "Asistio")
print("Segunda pregunta:", atributo2, reglas2)

# (3) Acierto del árbol en prueba
pred_arbol = [predecir_arbol(f, ganador, reglas_ganador, rama, atributo2, reglas2) for _, f in prueba.iterrows()]
print("Acierto del árbol en prueba:", round(acierto(pred_arbol), 3))

**✅ Compare:** su celda 📊 debe mostrar los **errores de cada rama**, la **rama impura**, la **segunda pregunta** y el **acierto del árbol en prueba**.

### ✍️ Responda

**1. Sus resultados:** ¿Cuál es la rama impura y cuál la segunda pregunta? Ordene ZeroR, OneR y el árbol según su acierto en prueba.

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Escriba el árbol como un **protocolo para la recepcionista**: ¿a qué dueños hay que llamar antes del control?

_Su respuesta:_

---
# Parte II · Ferretería online (35 puntos)

**Apriori.** La tabla `pedidos` tiene una fila por producto comprado (`Pedido`, `Producto`). Ya está convertida en **`canastas`**: una lista de **300 conjuntos**, uno por pedido. Por ejemplo, `{'Pintura', 'Brocha', 'Guantes'}`. Los 6 productos están en `PRODUCTOS`.

📄 **Antes de la tarea 4, lea la guía de apoyo «soporte(), confianza() y lift()».**

**Regresión.** La tabla `semanas` tiene 40 semanas con `Avisos` (lo que se gastó en avisos esa semana, en miles de $) y `Pedidos` (los pedidos de esa semana).

## Tarea 4 · Soporte: ¿qué se vende lo suficiente? (8 puntos)

**📖 Contexto.** La ferretería quiere recomendar productos («Otros clientes también llevaron…»). Primero mira **qué tan seguido** se vende cada uno.

> **💡 Concepto clave.** El **soporte** es la proporción de pedidos que contienen un producto (o varios juntos). Si un producto casi no se vende (menos de 10 %), no vale la pena buscar reglas con él.

### 🎯 Qué hacer

1. Lea la guía «soporte(), confianza() y lift()».
2. Complete `soporte()` siguiendo las pistas y ejecute la prueba 🧪.
3. Complete el `___` de la celda 📊.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| El **soporte de los 6 productos** y cuáles llegan al 10 %. | Muestra **qué tan populares** son los productos. Ojo: un producto que está en casi todos los pedidos puede **engañar** en la tarea 5. |

In [ ]:
# 🧩 Tarea 4 · complete el cuerpo de la(s) función(es)
def soporte(canastas, productos):
    """Soporte: en qué proporción de TODOS los pedidos aparecen juntos los productos dados.

    Recibe:   canastas  → lista de pedidos; cada pedido es un conjunto, p. ej. {"Pintura", "Guantes"}
              productos → lista con uno o más productos, p. ej. ["Pintura"] o ["Pintura", "Brocha"]
    Devuelve: un número entre 0 y 1
    Ejemplo:  con 4 pedidos, si "Pan" está en 3   →  soporte(canastas, ["Pan"]) = 0.75
              si "Pan" y "Queso" están juntos en 2  →  soporte(canastas, ["Pan", "Queso"]) = 0.5
    """
    buscados = set(productos)
    cuantos = 0
    for pedido in canastas:
        # Pista: «buscados <= pedido» es True si TODOS los productos buscados están en el pedido.
        # Si se cumple, sume 1 a cuantos.
        pass
    return ___                                   # cuántos pedidos cumplen, dividido por el total: len(canastas)

In [ ]:
# 🧪 Prueba de la tarea 4 con un ejemplo mínimo: ejecútela sin cambiarla
canastas_mini = [{"Pan", "Queso"}, {"Pan"}, {"Pan", "Queso", "Vino"}, {"Vino"}]
print(soporte(canastas_mini, ["Pan"]), soporte(canastas_mini, ["Pan", "Queso"]))
# ✅ Debe mostrar: 0.75 0.5

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`0.75 0.5`**: Pan está en 3 de los 4 pedidos del ejemplo; Pan y Queso juntos, en 2 de 4.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 4 · complete los ___ y ejecute
for p in PRODUCTOS:
    s = soporte(canastas, [___])                         # el soporte del producto p
    print(p, round(s, 3), "frecuente" if s >= 0.10 else "NO frecuente")

**✅ Compare:** su celda 📊 debe mostrar el **soporte de los 6 productos** y cuáles llegan al 10 %.

### ✍️ Responda

**1. Sus resultados:** ¿Qué producto no es frecuente? ¿Cuál aparece en casi todos los pedidos?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** ¿Vale la pena armar promociones con el producto no frecuente? ¿Por qué?

_Su respuesta:_

## Tarea 5 · Confianza y lift: ¿qué regla sirve para recomendar? (12 puntos)

**📖 Contexto.** Una regla **A → B** dice «quien lleva A también lleva B». La ferretería quiere saber **cuál usar para recomendar**.

> **💡 Concepto clave.** **Confianza**: de los pedidos con A, qué parte lleva B. **Lift** = confianza / soporte de B. **Lift mayor que 1**: la regla sirve. **Lift cercano a 1 o menor**: no sirve, aunque la confianza sea alta.

### 🎯 Qué hacer

1. Complete `confianza()` y `lift()` (use su `soporte()`) y ejecute la prueba 🧪.
2. Complete los `___` de la celda 📊 para las 3 reglas.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| El **soporte, la confianza y el lift** de 3 reglas: Pintura → Brocha, Brocha → Pintura y Rodillo → Guantes. | La confianza dice **qué tan seguido** se cumple la regla; el lift dice si **sirve para recomendar**. |

In [ ]:
# 🧩 Tarea 5 · complete el cuerpo de la(s) función(es)
def confianza(canastas, a, b):
    """Confianza de la regla a → b: de los pedidos que tienen a, qué proporción tiene también b.

    Confianza(a → b) = soporte(a y b) / soporte(a)      (se divide por el producto de la IZQUIERDA)
    Ejemplo:  si 3 pedidos tienen Pan y 2 de ellos también Queso  →  confianza(Pan → Queso) = 2/3
    """
    # confianza(a → b) = soporte de a y b JUNTOS / soporte de a
    # Pista: soporte(canastas, [a, b]) y soporte(canastas, [a])
    return ___


def lift(canastas, a, b):
    """Lift de la regla a → b: cuántas veces más se lleva b cuando se lleva a, comparado con el azar.

    Lift(a → b) = confianza(a → b) / soporte(b)
    Lectura:  mayor que 1 → asociación real · igual a 1 → no hay relación · menor que 1 → se evitan
    Ejemplo:  confianza(Pan → Queso) = 0.667 y Queso está en la mitad de los pedidos (0.5)
              →  lift = 0.667 / 0.5 = 1.33
    """
    # lift(a → b) = confianza(a → b) / soporte de b
    # Pista: confianza(canastas, a, b) y soporte(canastas, [b])
    return ___

In [ ]:
# 🧪 Prueba de la tarea 5 con un ejemplo mínimo: ejecútela sin cambiarla
canastas_mini = [{"Pan", "Queso"}, {"Pan"}, {"Pan", "Queso", "Vino"}, {"Vino"}]
print(round(confianza(canastas_mini, "Pan", "Queso"), 4), round(lift(canastas_mini, "Pan", "Queso"), 4))
# ✅ Debe mostrar: 0.6667 1.3333

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`0.6667 1.3333`**: de 3 pedidos con Pan, 2 llevan Queso (0,67); Queso está en la mitad de los pedidos → lift 0,67 / 0,5 = 1,33.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 5 · complete los ___ y ejecute
for a, b in [("Pintura", "Brocha"), ("Brocha", "Pintura"), ("Rodillo", "Guantes")]:
    print(a, "→", b,
          "| soporte:", round(soporte(canastas, [a, b]), 3),
          "| confianza:", round(___, 3),                 # use su función confianza
          "| lift:", round(___, 3))                      # use su función lift

**✅ Compare:** su celda 📊 debe mostrar el **soporte, la confianza y el lift** de 3 reglas: Pintura → Brocha, Brocha → Pintura y Rodillo → Guantes.

### ✍️ Responda

**1. Sus resultados:** ¿Qué regla tiene **más confianza**? ¿Y **más lift**? ¿Por qué no es la misma?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Si un cliente pone **Rodillo** en el carro, ¿conviene mostrarle Guantes? ¿Qué regla usaría para recomendar?

_Su respuesta:_

## Tarea 6 · Regresión: ¿cuántos pedidos traen los avisos? (15 puntos)

**📖 Contexto.** La ferretería anotó, durante 40 semanas, **cuánto gastó en avisos** en redes sociales (en miles de $) y **cuántos pedidos** recibió. Quiere saber cuántos pedidos esperar según lo que invierta.

> **💡 Concepto clave.** La **regresión lineal** dibuja la recta que mejor sigue a los puntos: **pedidos = a + b · avisos**. **b** (la pendiente) dice cuántos pedidos más llegan por cada $1.000 extra en avisos. La recta solo es confiable **dentro del rango** de gastos observados; usarla fuera es **extrapolar**.

### 🎯 Qué hacer

1. No hay función que completar.
2. Complete los `___` de la celda 📊: ajuste el modelo, muestre **a** y **b**, prediga los pedidos con **$60 mil** y muestre el gasto **mínimo y máximo** observado.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| Los números **a** y **b** de la recta, la **predicción con $60 mil** y el **rango** de gasto observado. | **b** es lo que rinde cada $1.000 en avisos, en pedidos reales. El rango dice **hasta dónde se puede confiar** en la recta. |

In [ ]:
# 📊 Tarea 6 · complete los ___ y ejecute
X = semanas[["Avisos"]]                                  # lo que se usa para predecir
y = semanas["Pedidos"]                                   # lo que se quiere predecir
modelo = LinearRegression().fit(___, ___)
print("a =", round(modelo.intercept_, 1))
print("b =", round(modelo.coef_[0], 2))
print("Pedidos esperados con $60 mil:", round(modelo.predict(pd.DataFrame({"Avisos": [___]}))[0], 1))
print("Gasto observado: de", semanas["Avisos"].min(), "a", semanas["Avisos"].max(), "mil")

**✅ Compare:** su celda 📊 debe mostrar los números **a** y **b** de la recta, la **predicción con $60 mil** y el **rango** de gasto observado.

### ✍️ Responda

**1. Sus resultados:** Escriba la recta con sus números. Explique al dueño qué significa **b**. ¿Cuántos pedidos se esperan con $60 mil?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** El dueño quiere gastar **$500 mil** en avisos y usar la recta para saber cuántos pedidos tendrá. ¿Qué le diría?

_Su respuesta:_

---
# Parte III · Liceo (30 puntos)

La tabla `liceo` tiene **60 estudiantes** con `Estudio` (horas de estudio a la semana) y `Actividad` (horas de actividad física a la semana). Ya están en **`puntos`**: una lista de pares `(x, y)`.

Se buscan **3 grupos** con k-means, partiendo de las estrellas **`ESTRELLAS_INICIO` = (1; 9), (2; 1) y (3; 3)**.

**Recuerde:** una **ronda** = **(1) elegir:** cada estudiante va a la estrella más cercana + **(2) moverse:** cada estrella va al promedio de su grupo. Se repite hasta que en una ronda **nadie cambie de grupo**.

## Tarea 7 · Una ronda de k-means: elegir y moverse (12 puntos)

**📖 Contexto.** La orientadora quiere formar **3 grupos de estudiantes parecidos** (según horas de estudio y de actividad física) para ofrecer talleres distintos.

> **💡 Concepto clave.** Cada grupo tiene una **estrella** (su centro). Una **ronda** tiene dos pasos: **elegir** (cada estudiante va a la estrella más cercana) y **moverse** (cada estrella va al promedio de su grupo).

### 🎯 Qué hacer

1. Complete `asignar()` (elegir) y `actualizar()` (moverse) siguiendo sus pistas.
2. Ejecute la prueba 🧪.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| Que la prueba 🧪 dé **exactamente** lo indicado. | Estas dos funciones **son una ronda completa**: en la tarea 8 se repiten hasta que los grupos no cambien. |

In [ ]:
# 🧩 Tarea 7 · complete el cuerpo de la(s) función(es)
def asignar(puntos, estrellas):
    """Paso 1 de una ronda (ELEGIR): cada punto elige la estrella más cercana.

    Recibe:   puntos    → lista de pares (x, y), p. ej. [(4, 9), (7, 4), ...]
              estrellas → lista de pares (x, y), una por grupo
    Devuelve: lista de grupos, uno por punto: 0 = primera estrella, 1 = segunda, ...
    Para comparar basta d² = (x − x★)² + (y − y★)², sin raíz.
    Ejemplo:  asignar([(1, 1), (8, 9)], [(3, 3), (10, 10)])  →  [0, 1]
    """
    grupos = []
    for x, y in puntos:
        d2 = [(x - ex) ** 2 + (y - ey) ** 2 for ex, ey in estrellas]   # d² a cada estrella
        # Pista: el grupo es la POSICIÓN de la menor distancia  →  d2.index(min(d2))
        grupos.append(___)
    return grupos


def actualizar(puntos, grupos, estrellas):
    """Paso 2 de una ronda (MOVERSE): cada estrella se va al promedio de los puntos de su grupo.

    Recibe:   puntos, grupos (lo que devuelve asignar) y las estrellas actuales
    Devuelve: lista de nuevas estrellas (x, y). Si un grupo quedara vacío, su estrella no se mueve.
    Ejemplo:  actualizar([(1, 1), (3, 1), (8, 9)], [0, 0, 1], [(0, 0), (10, 10)])  →  [(2.0, 1.0), (8.0, 9.0)]
    """
    nuevas = []
    for k in range(len(estrellas)):
        mios = [p for p, g in zip(puntos, grupos) if g == k]          # los puntos del grupo k
        if mios:
            x_prom = ___            # promedio de las x   (pista: sum(p[0] for p in mios) / len(mios))
            y_prom = ___            # promedio de las y
            nuevas.append((x_prom, y_prom))
        else:
            nuevas.append(estrellas[k])                               # grupo vacío: la estrella no se mueve
    return nuevas

In [ ]:
# 🧪 Prueba de la tarea 7 con un ejemplo mínimo: ejecútela sin cambiarla
puntos_mini = [(1, 1), (2, 1), (6, 6), (8, 9), (9, 8)]
print(asignar(puntos_mini, [(3, 3), (10, 10)]))
# ✅ Debe mostrar: [0, 0, 0, 1, 1]
puntos_mini = [(1, 1), (2, 1), (6, 6), (8, 9), (9, 8)]
nuevas = actualizar(puntos_mini, [0, 0, 0, 1, 1], [(3, 3), (10, 10)])
print([(round(float(x), 2), round(float(y), 2)) for x, y in nuevas])
# ✅ Debe mostrar: [(3.0, 2.67), (8.5, 8.5)]

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`[0, 0, 0, 1, 1]`**: con las estrellas en (3; 3) y (10; 10), los tres primeros puntos quedan más cerca de la primera estrella (grupo 0) y los dos últimos de la segunda (grupo 1).
- **`[(3.0, 2.67), (8.5, 8.5)]`**: la primera estrella va al promedio de (1; 1), (2; 1) y (6; 6) = (3; 2,67); la segunda, al de (8; 9) y (9; 8) = (8,5; 8,5).

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

## Tarea 8 · Repetir rondas y graficarlas (18 puntos)

**📖 Contexto.** Una ronda no basta: al moverse las estrellas, algunos estudiantes quedan más cerca de otra y **cambian de grupo**. Se repite hasta que nadie cambie.

> **💡 Concepto clave.** El algoritmo **termina** cuando en una ronda **nadie cambia de grupo**. Graficar cada ronda muestra cómo las estrellas **llegan al centro de cada grupo**.

### 🎯 Qué hacer

1. Complete los `___` de `graficar_ronda()` y de `kmeans_por_rondas()`: la estructura ya está armada.
2. Ejecute la prueba 🧪.
3. Ejecute la celda 📊: corre los 60 estudiantes y dibuja una gráfica por ronda.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| **Una gráfica por ronda**, cuántos estudiantes **cambian en cada ronda** (la última debe ser 0) y las **3 estrellas finales**. | Las estrellas finales son el **estudiante típico de cada grupo**: con ellas la orientadora decide los talleres. |

In [ ]:
# 🧩 Tarea 8 · complete el cuerpo de la(s) función(es)
def graficar_ronda(puntos, grupos, estrellas, titulo):
    """Dibuja UNA ronda: los puntos coloreados según su grupo y cada estrella como ★ grande.

    Requisitos de la gráfica:
      • cada grupo con un color distinto, y su estrella del MISMO color, con marcador "*" grande
      • ejes de 0 a 10, con nombre: "Horas de estudio" (x) y "Horas de actividad física" (y)
      • el título que se recibe (debe decir el número de ronda)
    No devuelve nada: solo muestra la gráfica (plt.show()).
    """
    colores = ["tab:red", "tab:blue", "tab:green"]
    plt.figure(figsize=(5, 5))
    for k, (ex, ey) in enumerate(estrellas):
        xs = [p[0] for p, g in zip(puntos, grupos) if g == k]         # las x de los puntos del grupo k
        ys = ___                                                      # las y de los puntos del grupo k
        plt.scatter(xs, ys, color=colores[k])                         # dibuja los puntos del grupo k
        plt.scatter([ex], [ey], color=colores[k], marker=___, s=400, edgecolor="black")   # la estrella: marker="*"
    plt.xlim(0, 10)
    plt.ylim(0, 10)
    plt.xlabel(___)                                                   # nombre del eje x
    plt.ylabel(___)                                                   # nombre del eje y
    plt.title(titulo)
    plt.show()


def kmeans_por_rondas(puntos, estrellas_inicio, graficar=True):
    """El CICLO de k-means: repite rondas (elegir + moverse) hasta que en una ronda NADIE cambie.

    En cada ronda:
      1. grupos = asignar(puntos, estrellas)
      2. se cuenta cuántos puntos cambiaron de grupo respecto de la ronda anterior
         (en la ronda 1 no hay con qué comparar: cambios = None)
      3. se guarda la ronda en el historial y, si graficar=True, se grafica con graficar_ronda()
      4. si cambios == 0 → se termina; si no → estrellas = actualizar(...) y se sigue
    Devuelve: (estrellas_finales, grupos_finales, historial)
              historial = lista con un diccionario por ronda: {"ronda": 1, "cambios": None, "estrellas": [...]}
    Ejemplo:  si en la ronda 2 cambia un punto y en la 3 ninguno  →  cambios [None, 1, 0]
    """
    estrellas = list(estrellas_inicio)
    anterior = None
    historial = []
    ronda = 0
    while True:
        ronda = ronda + 1
        grupos = ___                       # paso 1, ELEGIR: use asignar(puntos, estrellas)
        if anterior is None:
            cambios = None                 # en la ronda 1 no hay con qué comparar
        else:
            cambios = sum(a != b for a, b in zip(anterior, grupos))   # cuántos cambiaron de grupo
        historial.append({"ronda": ronda, "cambios": cambios, "estrellas": estrellas})
        if graficar:
            graficar_ronda(puntos, grupos, estrellas, f"Ronda {ronda} · cambios: {cambios}")
        if cambios == 0:
            return estrellas, grupos, historial     # nadie cambió: terminó
        estrellas = ___                    # paso 2, MOVERSE: use actualizar(puntos, grupos, estrellas)
        anterior = grupos

In [ ]:
# 🧪 Prueba de la tarea 8 con un ejemplo mínimo: ejecútela sin cambiarla
puntos_mini = [(1, 1), (2, 1), (6, 6), (8, 9), (9, 8)]
estrellas_m, grupos_m, historial_m = kmeans_por_rondas(puntos_mini, [(3, 3), (10, 10)], graficar=False)
print(len(historial_m), [h["cambios"] for h in historial_m])
# ✅ Debe mostrar: 3 [None, 1, 0]

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`3 [None, 1, 0]`**: 3 rondas: en la 1 no hay con qué comparar (None); en la 2 el punto (6; 6) cambia de grupo, porque las estrellas se movieron (1 cambio); en la 3 nadie cambia (0) y termina.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 8 · complete los ___ y ejecute
estrellas_fin, grupos_fin, historial = kmeans_por_rondas(puntos, ESTRELLAS_INICIO, graficar=True)
print("Cambios por ronda:", [h["cambios"] for h in historial])
print("Estrellas finales:", [(round(float(x), 2), round(float(y), 2)) for x, y in estrellas_fin])
print("Estudiantes por grupo:", [grupos_fin.count(k) for k in range(3)])

**✅ Compare:** su celda 📊 debe mostrar **Una gráfica por ronda**, cuántos estudiantes **cambian en cada ronda** (la última debe ser 0) y las **3 estrellas finales**.

### ✍️ Responda

**1. Sus resultados:** ¿En cuántas rondas termina y cuántos cambian en cada una? ¿Dónde quedan las estrellas?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Mirando las estrellas finales: ¿qué tipo de estudiante representa cada grupo? Póngales nombre y proponga **un taller** para uno.

_Su respuesta:_

---
## ✔️ Para entregar

1. **Entorno de ejecución → Reiniciar y ejecutar todo**: revise que todo corra sin errores.
2. **Archivo → Descargar → Descargar .ipynb**.
3. Súbalo al **Aula Virtual** como `Apellido_Nombre.ipynb`.